# Dataset overview

What the analyses work with: the ISMIR corpus (Section 1), the synthetic abstracts (Section 2) and the AI-use
disclosures found in the papers (Section 3). Dataset v2.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

from analysis.disclosure import TERMS, find_hits
from parsing.models import read_jsonl

JSONL = Path("../data/processed/ismir_2020_2025_v2.jsonl")
SYNTHETIC = Path("../data/interim/synthetic")
PERIODS = {2020: "before", 2021: "before", 2022: "before", 2023: "transition", 2024: "after", 2025: "after"}

papers = read_jsonl(JSONL)
meta = pd.DataFrame({
    "paper_id": [p.paper_id for p in papers],
    "year": [p.year for p in papers],
    "words": [p.n_words for p in papers],
    "abstract_words": [len(p.abstract.split()) for p in papers],
}).set_index("paper_id")
YEARS = sorted(meta.year.unique())
print(f"{len(meta)} papers, {meta.words.sum():,} words")

655 papers, 2,725,257 words


## 1. Corpus

Every paper in the ISMIR proceedings 2020–2025, downloaded from the conference pages on ismir.net and parsed with
GROBID. *Words* counts the full text (abstract and all sections, references excluded). ChatGPT was released on
30 Nov 2022, so 2020–2022 is the *before* period, 2023 a transition year and 2024–2025 the *after* period.

In [2]:
corpus = meta.groupby("year").agg(papers=("words", "size"), median_words=("words", "median"),
                                  median_abstract_words=("abstract_words", "median"))
corpus.insert(0, "period", corpus.index.map(PERIODS))
corpus.loc["total"] = ["", len(meta), meta.words.median(), meta.abstract_words.median()]
corpus.astype({"papers": int, "median_words": int, "median_abstract_words": int})

,period,papers,median_words,median_abstract_words
year,,,,
2020,before,115,4264,171
2021,before,104,4238,172
2022,before,113,4157,171
2023,transition,101,4100,163
2024,after,123,4158,172
2025,after,99,4030,174
total,,655,4145,171


## 2. Synthetic abstracts

One LLM-written abstract per paper, to compare with the human-written one. Each paper is sent to Claude in a new
Claude Code session (`claude -p`, script `analysis/generate.py`) with one user message: the prompt
*"Write an ISMIR-style abstract of about {N} words for the attached research paper."*, followed by the paper text
**without its abstract** (title and all sections). N is the original abstract's word count, so length does not
reveal the source. Settings: model `claude-sonnet-5-5`, effort `high`, thinking off, no tools, empty system prompt
and no project settings, so the model sees nothing but the paper. A leading "**Abstract**" heading, which the model
adds to most answers, is removed; the raw answer is kept in the file. The input text is dataset v1, which differs
from v2 only in the copyright footnote of 117 papers.

In [3]:
synth = pd.DataFrame([json.loads(f.read_text()) for f in sorted(SYNTHETIC.glob("*.json"))]).set_index("paper_id")
synth["synthetic_words"] = synth.abstract.str.split().str.len()
synth["original_words"] = synth.prompt.str.extract(r"about (\d+) words")[0].astype(int)
ratio = synth.synthetic_words / synth.original_words

print(f"{len(synth)} synthetic abstracts for {len(meta)} papers; missing: {sorted(set(meta.index) - set(synth.index))}")
for column in ["model", "effort", "claude_code_version"]:
    print(f"{column}: {', '.join(synth[column].unique())}")
print(f"generated {synth.generated_at.min()[:10]}; median {synth.input_tokens.median():,.0f} input "
      f"and {synth.output_tokens.median():,.0f} output tokens per paper")
print(f"length synthetic / original: median {ratio.median():.2f}, {ratio.between(0.8, 1.2).mean():.0%} within ±20%")
synth.groupby("year").agg(abstracts=("abstract", "size"), median_original_words=("original_words", "median"),
                          median_synthetic_words=("synthetic_words", "median"))

655 synthetic abstracts for 655 papers; missing: []
model: claude-sonnet-5-5
effort: high
claude_code_version: 2.1.316
generated 2026-11-03; median 9,154 input and 400 output tokens per paper
length synthetic / original: median 0.99, 100% within ±20%


,abstracts,median_original_words,median_synthetic_words
year,,,
2020,115,171.0,171.0
2021,104,172.0,171.5
2022,113,171.0,170.0
2023,101,163.0,159.0
2024,123,172.0,172.0
2025,99,174.0,173.0


## 3. AI-use disclosure

ISMIR did not require an AI-use statement in 2020–2025, so any disclosure is voluntary and may appear anywhere,
including acknowledgments and ethics statements. Two steps:

1. **Search** (`analysis/disclosure.py`): every sentence of the full text that names an AI tool (ChatGPT, GPT-n,
   Claude, Gemini, Bard, Copilot, Grammarly, DeepL, QuillBot), "LLM", "large language model", "generative AI", or
   "AI" near a writing verb (write, edit, proofread, polish, rephrase…).
2. **Check**: most hits are papers that *study* or *use* LLMs as part of the research, not for writing. We read every
   hit in an acknowledgment, ethics or appendix section, and every hit where the sentence also mentions writing,
   editing or translating. Labels: `used_ai` (the authors used AI to write), `no_ai_statement` (they state they did
   not), `not_disclosure` (everything else).

In [4]:
hits = pd.DataFrame([row for p in papers for row in find_hits(p)])
WRITING = re.compile(r"writ|edit|proofread|polish|grammar|rephras|paraphras|manuscript|translat", re.I)
to_check = hits[hits.section_labels.str.contains("acknowledgments|ethics|appendix")
                | hits.sentence.str.contains(WRITING)]

# Result of reading the sentences in `to_check`; every other paper is not_disclosure.
LABELS = {"2025_000087": "no_ai_statement"}

print(f"{len(hits)} sentences in {hits.paper_id.nunique()} papers name an AI tool; {len(to_check)} sentences "
      f"in {to_check.paper_id.nunique()} papers checked")
with pd.option_context("display.max_colwidth", 140):
    display(to_check[["paper_id", "section", "sentence"]].assign(label=to_check.paper_id.map(LABELS).fillna("not_disclosure")))

800 sentences in 82 papers name an AI tool; 45 sentences in 28 papers checked


,paper_id,section,sentence,label
16,2020_000167,RELATED WORK,"For example, an AI might add to a user's drawing, alternate writing sentences of a story with a human, or auto-complete missing parts of...",not_disclosure
36,2022_000017,EVALUATION,"Our attribution visualization showed the difference between notes added by the AI and edited notes, which helped P3: awhere was the last...",not_disclosure
62,2023_000014,DESIGN REQUIREMENTS,"Here, since the remotelyparticipated volunteers were Japanese speakers recruited via word-of-mouth communication, we told them that they...",not_disclosure
66,2023_000014,Implementation,"In addition, we used DeepL API to translate text prompts into English when they were provided in non-English languages because we observ...",not_disclosure
137,2023_000048,Large Language Model for Data Generation,The list of tags are appended with a carefully written task instruction as an input (prompt) to a large language model.,not_disclosure
151,2024_000005,ETHICS STATEMENT,"Additionally, using FruitsMusic to construct or trigger other generative AI techniques, such as lyrics and music generation, may violate...",not_disclosure
153,2024_000007,abstract,"Yet scant research explores the details of how these LLMs perform on advanced music understanding and conditioned generation, especially...",not_disclosure
225,2024_000014,Synthetic Lyrics Generation for Moral Assessment,"We tasked GPT-4 with generating lyrics by formulating a prompt, as follows: Prompt: You are an assistant to a songwriter, you need to as...",not_disclosure
244,2024_000014,ETHICS STATEMENT,"In this study, we employed large language models (LLMs) to generate synthetic lyrics.",not_disclosure
245,2024_000014,ETHICS STATEMENT,"While powerfull language models like BERT and GPT-4 offer significant potential to enhance communication and support social campaigns, t...",not_disclosure


In [5]:
labels = pd.Series(LABELS)
disclosure = pd.DataFrame({
    "papers": meta.groupby("year").size(),
    "mention_AI_tool": hits.groupby("year").paper_id.nunique(),
    "used_ai": labels[labels == "used_ai"].index.str[:4].astype(int).value_counts(),
    "no_ai_statement": labels[labels == "no_ai_statement"].index.str[:4].astype(int).value_counts(),
}).reindex(YEARS).fillna(0).astype(int)
disclosure.loc["total"] = disclosure.sum()
disclosure

,papers,mention_AI_tool,used_ai,no_ai_statement
2020,115,3,0,0
2021,104,3,0,0
2022,113,6,0,0
2023,101,7,0,0
2024,123,34,0,0
2025,99,29,0,1
total,655,82,0,1


## Summary

- 655 papers (2020–2022: 332, 2023: 101, 2024–2025: 222), median 4,145 words each (2.7 million in total).
- 655 synthetic abstracts, one per paper, by `claude-sonnet-5-5`, matched to the original length.
- AI tools are named in 82 papers (63 of them from 2024–2025), but always as a research topic or method. No paper
  discloses using AI to write; one 2025 paper states that it did not (and used code generation for figures).